# Hadamard-coin quantum-walk QNN on the discrete-log concept class (Liu–Arunachalam–Temme 2021)

**What this runs.** A simple quantum-walk QNN, built on the structure of IBM's *Quantum variational circuits / QNN* lesson
(feature map → trainable part → expectation value → MSE loss), is tested on the discrete-log (DLP) learning problem of
Liu, Arunachalam & Temme, *Nat. Phys.* 17, 1013 (2021). It is compared with LAT's own quantum kernel, qml-benchmarks
quantum models (IQP kernel, projected quantum kernel, data re-uploading), classical models (MLP, RBF-SVM, random forest)
and a no-walk control, all under one protocol.

**Data.** $p$ is a prime with $n$ bits, $g$ a generator of $\mathbb Z_p^*$, $x\in\mathbb Z_p^*$, $a=\log_g x$.
Label $y=+1$ iff $a\in[s,\,s+\tfrac{p-3}{2}]$ (a half-circle of exponents; $s$ random per draw).
150 train / 100 test distinct $x$ per draw, 5 draws per $n\in\{8,10,12,16\}$.

**Model (QW-QNN).**
1. *Discrete-log encoding* $|x\rangle\mapsto|a\rangle$ (Shor's DLP subroutine, as in LAT; computed classically in simulation).
2. *Walk:* coin qubit $(|0\rangle+i|1\rangle)/\sqrt2$ and an $h$-qubit position register holding the top $h$ bits of $a$
   (a vertex of the cycle $C_{2^h}$); $t$ steps of $W=S\,(H\otimes I)$, $S=|0\rangle\langle0|\otimes\mathrm{INC}+|1\rangle\langle1|\otimes\mathrm{DEC}$.
3. *Readout:* measure the position, $f(a)=\sum_u w_u P_t(u\mid a)+b$, i.e. the expectation of a trainable diagonal
   observable $O_w$; label $\operatorname{sgn} f$. Loss: the lesson's MSE; it is linear in $(w,b)$, so the exact (ridge) minimiser is used.
   The walk circuit is checked against Qiskit (`build_walk_circuit`).
4. *Continuous-time variant* (`CTWalkQNN`): the coined walk replaced by $e^{-itA}$ on the same cycle, $A$ its adjacency
   matrix, $t$ continuous; implemented as QFT$^\dagger$ → diagonal phase → QFT and checked against Qiskit (`build_ctqw_circuit`).
   Everything else (encoding, read-out, loss, protocol) is identical.

**Protocol.** Hyperparameters by 5-fold CV on the training set only; each test set scored once; qml-benchmarks grids
for C / SVC / MLP (data re-uploading grid reduced to 8 settings for run time); suite quantum models run up to $n=12$ qubits.
**Every model receives the same input, $x$.** The quantum pipelines apply Shor's discrete-log circuit to $x$ as their
first layer (simulated with a lookup table); the raw records tag this `input: "log x"`. The report prints a main table
(every model on $x$) and an ablation table: the walk removed ($t=0$), and classical models handed $\log x$ directly,
which a classical computer cannot compute efficiently at large $n$.

**Scope — read before citing the results.** (i) Everything that separates the quantum models from the classical ones
comes from the DLP step; the ablation rows with classical models handed $\log x$ show this directly. (ii) At $n\le16$ a classical brute-force discrete log
is trivial, so the gap over classical models is against generic ML models, and it is asymptotic only under DLP hardness.
(iii) The no-walk control and LAT's kernel receive the same DLP encoding, so they are the fair test of whether the walk adds anything.

**How to run (Colab).** *Runtime → Run all.* The first run installs pinned packages (PennyLane 0.34 / JAX 0.4.23, the versions
qml-benchmarks needs); when it says so, *Runtime → Restart session* and *Run all* again. Results are saved to Google Drive
after every model, so a disconnect loses at most one fit and re-running resumes. A CPU runtime is enough; the qml-benchmarks
quantum models are skipped by default (`SKIP_MODELS`; hours at 10–12 qubits). Set `SMOKE = True` for a ~1-minute check.


In [ ]:
# ---- configuration ------------------------------------------------------------------------
SMOKE = False                 # True = tiny end-to-end test (n=8, 1 draw, heavy suite models skipped)
USE_DRIVE = True              # save results to Google Drive (recommended on Colab; resumes after a disconnect)
N_BITS = [8, 10, 12, 16]      # bit-length of the prime p
MAX_BITS_SUITE_QUANTUM = 12   # qml-benchmarks quantum models (n qubits each) are run up to this size
DRAWS = [0, 1, 2, 3, 4]       # independent draws of (offset s, train/test split)
M_TRAIN, M_TEST = 150, 100    # distinct x values; train and test never share an x
# qml-benchmarks quantum models (IQP kernel, projected kernel, data re-uploading) are not run by default:
# the run was stopped after 5 fits at n = 8 (hours per size at 10-12 qubits). Set to set() to include them.
SKIP_MODELS = {"IQP kernel", "Projected quantum kernel", "Data re-uploading"}
if SMOKE:
    N_BITS, DRAWS = [8], [0]
    SKIP_MODELS = {"IQP kernel", "Projected quantum kernel", "Data re-uploading"}


In [ ]:
# ---- setup: pinned environment matching qml-benchmarks (PennyLane 0.34, JAX 0.4.23) ------------------
# On Colab this installs into the runtime; locally, run the notebook inside a venv.
import os, sys, subprocess
PINS = ["numpy<2", "scipy==1.11.4", "scikit-learn>=1.3,<1.10", "pennylane==0.34.0", "autoray==0.6.8",
        "jax[cpu]==0.4.23", "jaxlib==0.4.23", "ml_dtypes<0.5", "chex==0.1.85", "optax==0.1.7", "flax==0.7.5",
        "orbax-checkpoint==0.4.8", "tensorstore<0.1.60", "numpyro==0.14.0", "tqdm", "pandas", "matplotlib", "qiskit>=2"]
try:
    import pennylane, jax
    ok = pennylane.__version__ == "0.34.0" and jax.__version__ == "0.4.23"
except ImportError:
    ok = False
if not ok:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    print("Installed pinned packages. On Colab: Runtime -> Restart session, then Run all again.")
REPO = os.environ.get("QMLB_REPO", "qml-benchmarks")
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/XanaduAI/qml-benchmarks.git", REPO], check=True)
sys.path.insert(0, os.path.join(REPO, "src"))

OUT_DIR = "."
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/qw_qnn_lat"
    except Exception as e:
        print("Drive not available, saving locally:", e)
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS = os.path.join(OUT_DIR, "lat_results_smoke.json" if SMOKE else "lat_results.json")
print("results file:", RESULTS)


In [ ]:
# ---- data: Liu-Arunachalam-Temme (2021) discrete-log concept class ------------------
# x in Z_p^*,  a = log_g x,  label y = +1 iff a in [s, s + (p-3)/2]  (a half-circle of exponents)
import numpy as np

PRIMES = {8: 251, 10: 1021, 12: 4093, 14: 16381, 16: 65521}   # largest n-bit primes


def primitive_root(p):
    phi = p - 1
    fac, m, d = [], phi, 2
    while d * d <= m:
        if m % d == 0:
            fac.append(d)
            while m % d == 0:
                m //= d
        d += 1
    if m > 1:
        fac.append(m)
    for g in range(2, p):
        if all(pow(g, phi // q, p) != 1 for q in fac):
            return g


def dlog_table(p, g):
    """log_g for every x in Z_p^*. This brute-force table stands in for Shor's discrete-log
    subroutine in simulation; it is exponential in n classically, which is the point of the dataset."""
    log = np.zeros(p, dtype=np.int64)
    v = 1
    for a in range(p - 1):
        log[v] = a
        v = v * g % p
    return log


def bits(v, n):
    return (((np.asarray(v)[:, None] >> np.arange(n)) & 1) * 2 - 1).astype(float)   # +-1 features


def make_lat(n, draw, m_train, m_test):
    """One draw: random offset s, disjoint train/test sets of distinct x (no memorisation possible)."""
    p = PRIMES[n]
    g = primitive_root(p)
    log = dlog_table(p, g)
    rng = np.random.default_rng(1000 * n + draw)
    s = int(rng.integers(p - 1))
    xs = rng.choice(np.arange(1, p), size=m_train + m_test, replace=False)
    a = log[xs]
    y = np.where((a - s) % (p - 1) <= (p - 3) // 2, 1, -1)
    tr, te = slice(0, m_train), slice(m_train, None)
    return dict(n=n, p=p, g=g, s=s, x_tr=xs[tr], x_te=xs[te], a_tr=a[tr], a_te=a[te], y_tr=y[tr], y_te=y[te])


In [ ]:
# ---- models ---------------------------------------------------------------------------
# Hadamard-coin QW-QNN. Input: the exponent a = log_g x (from the discrete-log step).
#   position register: h qubits holding the top h bits of a  ->  vertex v = floor(a 2^h/(p-1)) of the cycle C_{2^h}
#   coin: 1 qubit, initialised (|0> + i|1>)/sqrt2
#   t steps of  W = S (H_coin (x) I),  S = |0><0| (x) INC + |1><1| (x) DEC   (Hadamard walk on C_{2^h})
#   readout: measure position, f(a) = sum_u w_u P_t(u | v) + b   (trainable observable O_w = sum_u w_u |u><u|)
#   training: the IBM lesson's MSE loss on y in {-1,+1}; f is linear in (w, b), so the minimiser is exact (ridge)
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.svm import SVC as _SVC

H2 = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)


def walk_distribution(h, t):
    """P_t(u | start 0) for the Hadamard walk on C_{2^h}; translation invariance gives P_t(u|v) = P_t(u-v|0)."""
    N = 2 ** h
    psi = np.zeros((2, N), complex)
    psi[:, 0] = np.array([1, 1j]) / np.sqrt(2)
    for _ in range(t):
        psi = H2 @ psi
        psi = np.stack([np.roll(psi[0], 1), np.roll(psi[1], -1)])   # coin 0 -> v+1, coin 1 -> v-1
    return (np.abs(psi) ** 2).sum(0)


class HadamardWalkQNN(BaseEstimator, ClassifierMixin):
    """X: column vector of exponents a (ints). t = 0 is the no-walk control (readout of the start vertex only)."""

    def __init__(self, p=251, h=4, t=4, ridge=1e-2, shots=None, random_state=0):
        self.p, self.h, self.t, self.ridge, self.shots, self.random_state = p, h, t, ridge, shots, random_state

    def _probs(self, X):
        a = np.asarray(X).reshape(-1).astype(np.int64)
        N = 2 ** self.h
        v = (a * N) // (self.p - 1)
        P0 = walk_distribution(self.h, self.t)
        idx = (np.arange(N)[None, :] - v[:, None]) % N
        return P0[idx]                                           # (B, N): P_t(u | v)

    def fit(self, X, y):
        y = np.asarray(y, float)
        self.classes_ = np.array([-1, 1])
        Phi = np.c_[self._probs(X), np.ones(len(y))]
        R = self.ridge * np.eye(Phi.shape[1]); R[-1, -1] = 0
        self.coef_ = np.linalg.solve(Phi.T @ Phi + R, Phi.T @ y)
        return self

    def decision_function(self, X):
        P = self._probs(X)
        if self.shots:                                           # finite-shot estimate of <O_w>
            rng = np.random.default_rng(self.random_state)
            P = np.array([rng.multinomial(self.shots, q / q.sum()) for q in P]) / self.shots
        return P @ self.coef_[:-1] + self.coef_[-1]

    def predict(self, X):
        return np.where(self.decision_function(X) >= 0, 1, -1)


class LATKernelSVM(BaseEstimator, ClassifierMixin):
    """Liu-Arunachalam-Temme quantum kernel: |phi(x)> = 2^{-k/2} sum_{i<2^k} |x g^i>,
    K = |<phi(x)|phi(x')>|^2 = (max(0, 2^k - d(a,a')) / 2^k)^2,  d = forward cyclic distance in exponents
    (the overlap of two length-2^k intervals depends on the circular distance)."""

    def __init__(self, p=251, k=6, C=1.0):
        self.p, self.k, self.C = p, k, C

    def _K(self, A, B):
        A = np.asarray(A).reshape(-1, 1); B = np.asarray(B).reshape(1, -1)
        d = np.abs(A - B) % (self.p - 1)
        d = np.minimum(d, self.p - 1 - d)
        W = 2 ** self.k
        return (np.maximum(0, W - d) / W) ** 2

    def fit(self, X, y):
        self.X_ = np.asarray(X).reshape(-1)
        self.classes_ = np.array([-1, 1])
        self.svc_ = _SVC(kernel="precomputed", C=self.C).fit(self._K(self.X_, self.X_), y)
        return self

    def predict(self, X):
        return self.svc_.predict(self._K(X, self.X_))


def build_walk_circuit(h, t, v):
    """The walk as a Qiskit circuit (qubit 0 = coin, qubits 1..h = position, little-endian), for checking."""
    from qiskit import QuantumCircuit
    inc = QuantumCircuit(h, name="INC")
    for j in reversed(range(1, h)):
        inc.mcx(list(range(j)), j)
    inc.x(0)
    INC0 = inc.to_gate().control(1, ctrl_state=0)
    DEC1 = inc.inverse().to_gate().control(1, ctrl_state=1)
    qc = QuantumCircuit(1 + h)
    for j in range(h):
        if (v >> j) & 1:
            qc.x(1 + j)
    qc.h(0); qc.s(0)                       # coin (|0> + i|1>)/sqrt2
    for _ in range(t):
        qc.h(0)
        qc.append(INC0, list(range(1 + h)))
        qc.append(DEC1, list(range(1 + h)))
    return qc


# ---- continuous-time variant: CTQW on the same cycle, same read-out -------------------------------------
# |psi_t> = exp(-i t A) |v>,  A = adjacency of C_{2^h};  f(a) = sum_u w_u |<u|exp(-i t A)|v>|^2 + b
# exp(-itA) = F diag(exp(-2it cos(2 pi k/N))) F^dagger  (F = QFT): a QFT, a diagonal phase, an inverse QFT.
def ctqw_distribution(h, t):
    N = 2 ** h
    k = np.arange(N)
    lam = 2 * np.cos(2 * np.pi * k / N)
    amp = np.fft.ifft(np.exp(-1j * t * lam))          # <u|e^{-itA}|0> = (1/N) sum_k e^{2 pi i u k/N} e^{-it lam_k}
    return np.abs(amp) ** 2


class CTWalkQNN(HadamardWalkQNN):
    """Same model as HadamardWalkQNN with the coined walk replaced by exp(-i t A_{C_{2^h}}); t is continuous."""

    def _probs(self, X):
        a = np.asarray(X).reshape(-1).astype(np.int64)
        N = 2 ** self.h
        v = (a * N) // (self.p - 1)
        P0 = ctqw_distribution(self.h, self.t)
        return P0[(np.arange(N)[None, :] - v[:, None]) % N]

    def fit(self, X, y):
        # same least-squares problem (ridge on w, unpenalised bias); sklearn switches to the dual form when 2^h > samples
        from sklearn.linear_model import Ridge
        self.classes_ = np.array([-1, 1])
        r = Ridge(alpha=self.ridge).fit(self._probs(X), np.asarray(y, float))
        self.coef_ = np.r_[r.coef_, r.intercept_]
        return self


def build_ctqw_circuit(h, t, v):
    """exp(-itA) on the cycle as QFT^dagger . diag . QFT on h qubits (little-endian), for checking."""
    from qiskit import QuantumCircuit
    from qiskit.circuit.library import QFTGate, DiagonalGate
    N = 2 ** h
    qc = QuantumCircuit(h)
    for j in range(h):
        if (v >> j) & 1:
            qc.x(j)
    qc.append(QFTGate(h).inverse(), range(h))
    qc.append(DiagonalGate(list(np.exp(-1j * t * 2 * np.cos(2 * np.pi * np.arange(N) / N)))), range(h))
    qc.append(QFTGate(h), range(h))
    return qc


In [ ]:
# ---- check: the numpy walk equals the Qiskit circuit ------------------------------------------
from qiskit.quantum_info import Statevector
err = 0.0
for h in [3, 4, 5]:
    for t in [0, 1, 3, 7, 12]:
        for v in [0, 5, 2 ** h - 1]:
            pq = Statevector(build_walk_circuit(h, t, v)).probabilities().reshape(2 ** h, 2).sum(1)  # index c + 2u
            pn = walk_distribution(h, t)[(np.arange(2 ** h) - v) % 2 ** h]
            err = max(err, np.abs(pq - pn).max())
print(f"max |P_qiskit - P_numpy| = {err:.1e}")
assert err < 1e-10
print(build_walk_circuit(3, 1, 5).draw(output="text", fold=120))

# CTQW: numpy distribution vs scipy expm vs the Qiskit QFT-diagonal-QFT circuit
from scipy.linalg import expm
err_c = 0.0
for h in [3, 4, 5]:
    N = 2 ** h
    A = np.roll(np.eye(N), 1, 0) + np.roll(np.eye(N), -1, 0)
    for t in [0.0, 0.5, 1.7, 4.0, 9.3]:
        for v in [0, 5, N - 1]:
            pe = np.abs(expm(-1j * t * A)[:, v]) ** 2
            pn = ctqw_distribution(h, t)[(np.arange(N) - v) % N]
            pq = Statevector(build_ctqw_circuit(h, t, v)).probabilities()
            err_c = max(err_c, np.abs(pe - pn).max(), np.abs(pe - pq).max())
print(f"CTQW: max |P_expm - P_numpy|, |P_expm - P_qiskit| = {err_c:.1e}")
assert err_c < 1e-10


In [ ]:
# ---- qml-benchmarks models (the suite's own implementations) --------------------------
# Kernel models: the Gram matrix for each non-C hyperparameter setting is computed once on the training set
# (and once test x train) with the model's own precompute_kernel, then C is cross-validated on slices of it.
# For +-1 bit features the suite's MinMaxScaler is identical on every fold, so the slices equal what refitting
# would compute (for PQK, gamma's data-dependent default is taken from the full training set).
import itertools, warnings
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC as SkSVC
from sklearn.ensemble import RandomForestClassifier
import qml_benchmarks.models as QB

CV = StratifiedKFold(5, shuffle=True, random_state=42)
C_GRID = [0.1, 1, 10, 100]                                    # suite grid for C


def _kernel_arm(make_model, settings, Xtr, ytr, Xte):
    best = (-1, None, None)
    for st in settings:
        m = make_model(**st)
        m.initialize(Xtr.shape[1], np.array([-1, 1]))
        m.scaler = MinMaxScaler(feature_range=(-np.pi / 2, np.pi / 2)).fit(Xtr)
        A, B = m.transform(Xtr), m.transform(Xte)
        Ktr = np.asarray(m.precompute_kernel(A, A)); Kte = np.asarray(m.precompute_kernel(B, A))
        for C in C_GRID:
            acc = np.mean([(SkSVC(kernel="precomputed", C=C).fit(Ktr[np.ix_(i, i)], ytr[i])
                            .predict(Ktr[np.ix_(j, i)]) == ytr[j]).mean() for i, j in CV.split(Xtr, ytr)])
            if acc > best[0]:
                best = (acc, dict(st, C=C), (Ktr, Kte))
    acc, hp, (Ktr, Kte) = best
    pred = SkSVC(kernel="precomputed", C=hp["C"]).fit(Ktr, ytr).predict(Kte)
    return acc, hp, pred


def iqp_kernel(Xtr, ytr, Xte):
    return _kernel_arm(lambda **k: QB.IQPKernelClassifier(jit=True, **k),
                       [{"repeats": r} for r in [1, 5, 10]], Xtr, ytr, Xte)


def projected_kernel(Xtr, ytr, Xte):
    sets = [{"gamma_factor": g, "t": t, "trotter_steps": s}
            for g, t, s in itertools.product([0.1, 1, 10], [0.01, 0.1, 1.0], [1, 3, 5])]
    return _kernel_arm(lambda **k: QB.ProjectedQuantumKernel(**k), sets, Xtr, ytr, Xte)


def _gridsearch(est, grid, Xtr, ytr, Xte):
    gs = GridSearchCV(est, grid, cv=CV, n_jobs=1).fit(Xtr, ytr)
    return gs.best_score_, {k: (v if np.isscalar(v) else str(v)) for k, v in gs.best_params_.items()}, gs.predict(Xte)


def data_reuploading(Xtr, ytr, Xte):
    # reduced from the suite grid (3 lr x 4 layers x 3 observables) to keep the run tractable
    grid = {"learning_rate": [0.01, 0.1], "n_layers": [1, 5], "observable_type": ["half", "full"]}
    return _gridsearch(QB.DataReuploadingClassifier(random_state=42), grid, Xtr, ytr, Xte)


def mlp(Xtr, ytr, Xte):
    grid = {"learning_rate_init": [0.001, 0.01, 0.1],
            "hidden_layer_sizes": [(100,), (10, 10, 10, 10), (50, 10, 5)], "alpha": [0.01, 0.001, 0.0001]}
    return _gridsearch(QB.MLPClassifier(random_state=42), grid, Xtr, ytr, Xte)


def svc_rbf(Xtr, ytr, Xte):
    return _gridsearch(QB.SVC(), {"gamma": [0.001, 0.01, 0.1, 1], "C": C_GRID}, Xtr, ytr, Xte)


def random_forest(Xtr, ytr, Xte):
    grid = {"n_estimators": [100, 300], "max_depth": [None, 4, 8]}
    return _gridsearch(RandomForestClassifier(random_state=42), grid, Xtr, ytr, Xte)


def walk_qnn(atr, ytr, ate, p, walk=True):
    grid = {"h": [3, 4, 5, 6], "t": [1, 2, 3, 4, 6, 8, 12, 16] if walk else [0],
            "ridge": [1e-3, 1e-2, 1e-1, 1.0]}
    return _gridsearch(HadamardWalkQNN(p=p), grid, atr.reshape(-1, 1), ytr, ate.reshape(-1, 1))


def lat_kernel(atr, ytr, ate, p, n):
    grid = {"k": [n - 2, n - 3, n - 4, n - 5, n - 6], "C": C_GRID}
    return _gridsearch(LATKernelSVM(p=p), grid, atr.reshape(-1, 1), ytr, ate.reshape(-1, 1))


def ct_walk_qnn(atr, ytr, ate, p):
    grid = {"h": [3, 4, 5, 6], "t": [0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0],
            "ridge": [1e-3, 1e-2, 1e-1, 1.0]}
    return _gridsearch(CTWalkQNN(p=p), grid, atr.reshape(-1, 1), ytr, ate.reshape(-1, 1))


# ---- uncapped setup (PREREGISTRATION_2): walk resolution h in {3..n}; LAT kernel over every width -----------------
# CV uses the exact dual (kernel-ridge) form of CTWalkQNN.fit (identical selection to GridSearchCV; checked on old draws)
FULL_T = [0.1, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8]
FULL_RIDGE = [1e-3, 1e-2, 1e-1, 1.0]


def _dual_fit_predict(K_tr, K_te, y, lam):
    m_tr = K_tr.mean(1); mm = K_tr.mean()
    Kc = K_tr - m_tr[:, None] - m_tr[None, :] + mm
    alpha = np.linalg.solve(Kc + lam * np.eye(len(y)), y - y.mean())
    Kt = K_te - K_te.mean(1)[:, None] - m_tr[None, :] + mm
    return Kt @ alpha + y.mean()


def ct_walk_qnn_full(atr, ytr, ate, p, n, t_grid=FULL_T):
    y = ytr.astype(float)
    splits = list(CV.split(atr.reshape(-1, 1), y))
    best = (-1, None)
    for h in range(3, n + 1):
        for t in t_grid:
            P = CTWalkQNN(p=p, h=h, t=t)._probs(atr.reshape(-1, 1)); K = P @ P.T
            for lam in FULL_RIDGE:
                acc = np.mean([(np.where(_dual_fit_predict(K[np.ix_(i, i)], K[np.ix_(j, i)], y[i], lam) >= 0, 1, -1)
                                == y[j]).mean() for i, j in splits])
                key = (h, lam, t)
                if acc > best[0] + 1e-12 or (abs(acc - best[0]) <= 1e-12 and key < best[1]):
                    best = (acc, key)
    acc, (h, lam, t) = best
    pred = CTWalkQNN(p=p, h=h, t=t, ridge=lam).fit(atr.reshape(-1, 1), y).predict(ate.reshape(-1, 1))
    return acc, {"h": h, "t": t, "ridge": lam}, pred


def lat_kernel_full(atr, ytr, ate, p, n):
    grid = {"k": list(range(1, n)), "C": [0.01, 0.1, 1, 10, 100, 1000]}
    return _gridsearch(LATKernelSVM(p=p), grid, atr.reshape(-1, 1), ytr, ate.reshape(-1, 1))


In [ ]:
# ---- protocol: for each (n, draw): 5-fold CV on the training set -> refit -> test set scored once. Resumable. --
import json, os, time, warnings
warnings.filterwarnings("ignore")

# model name -> (input, function, which sizes)
def model_table(n):
    heavy = n <= MAX_BITS_SUITE_QUANTUM
    T = {
        # what a classical learner is given in LAT: the bits of x
        ("x", "MLP"): mlp, ("x", "SVC (RBF)"): svc_rbf, ("x", "Random forest"): random_forest,
        # discrete-log encoded input a = log_g x
        ("log x", "QW-QNN (Hadamard walk)"): "walk", ("log x", "QW-QNN (continuous-time walk)"): "ctwalk", ("log x", "No-walk control (t=0)"): "nowalk",
        ("log x", "LAT kernel"): "lat",
        # uncapped setup (used for the thesis table and figure)
        ("log x", "QW-QNN (continuous-time walk, full resolution)"): "ctfull",
        ("log x", "LAT kernel (full grid)"): "latfull",
        ("log x", "No-walk control (full resolution)"): "nowalkfull",
        ("log x", "MLP (reference)"): mlp, ("log x", "SVC (RBF) (reference)"): svc_rbf,
        ("log x", "Random forest (reference)"): random_forest,
    }
    if heavy:
        for inp in ["x", "log x"]:
            T[(inp, "IQP kernel")] = iqp_kernel
            T[(inp, "Projected quantum kernel")] = projected_kernel
            T[(inp, "Data re-uploading")] = data_reuploading
    return T


HEAVY = {"IQP kernel", "Projected quantum kernel", "Data re-uploading"}
results = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
# pass 1: every fast model on every size/draw; pass 2: the slow qml-benchmarks quantum models
for n, draw, heavy_pass in [(n, dr, hp) for hp in (False, True) for n in N_BITS for dr in DRAWS]:
    if True:
        d = make_lat(n, draw, M_TRAIN, M_TEST)
        feats = {"x": (bits(d["x_tr"], n), bits(d["x_te"], n)), "log x": (bits(d["a_tr"], n), bits(d["a_te"], n))}
        for (inp, name), fn in model_table(n).items():
            key = f"{n}|{draw}|{inp}|{name}"
            if key in results or (name in SKIP_MODELS) or ((name in HEAVY) != heavy_pass):
                continue
            t0 = time.time()
            if fn == "walk":
                cv_acc, hp, pred = walk_qnn(d["a_tr"], d["y_tr"], d["a_te"], d["p"], walk=True)
            elif fn == "ctwalk":
                cv_acc, hp, pred = ct_walk_qnn(d["a_tr"], d["y_tr"], d["a_te"], d["p"])
            elif fn == "ctfull":
                cv_acc, hp, pred = ct_walk_qnn_full(d["a_tr"], d["y_tr"], d["a_te"], d["p"], n)
            elif fn == "nowalkfull":
                cv_acc, hp, pred = ct_walk_qnn_full(d["a_tr"], d["y_tr"], d["a_te"], d["p"], n, t_grid=[0.0])
            elif fn == "latfull":
                cv_acc, hp, pred = lat_kernel_full(d["a_tr"], d["y_tr"], d["a_te"], d["p"], n)
            elif fn == "nowalk":
                cv_acc, hp, pred = walk_qnn(d["a_tr"], d["y_tr"], d["a_te"], d["p"], walk=False)
            elif fn == "lat":
                cv_acc, hp, pred = lat_kernel(d["a_tr"], d["y_tr"], d["a_te"], d["p"], n)
            else:
                cv_acc, hp, pred = fn(feats[inp][0], d["y_tr"], feats[inp][1])
            rec = {"n": n, "draw": draw, "input": inp, "model": name, "cv_acc": float(cv_acc),
                   "test_acc": float(np.mean(pred == d["y_te"])), "hp": {k: (v.item() if hasattr(v, "item") else v) for k, v in hp.items()},
                   "correct": [int(c) for c in (pred == d["y_te"])], "secs": round(time.time() - t0, 1)}
            if fn in ("walk", "ctwalk", "ctfull"):   # the selected model evaluated from finite measurement shots
                M = HadamardWalkQNN if fn == "walk" else CTWalkQNN
                hp = {k: v for k, v in hp.items()}
                rec["test_acc_shots"] = {str(S): float(np.mean(M(p=d["p"], shots=S, **hp).fit(
                    d["a_tr"].reshape(-1, 1), d["y_tr"]).predict(d["a_te"].reshape(-1, 1)) == d["y_te"]))
                    for S in [100, 1000]}
            results[key] = rec
            json.dump(results, open(RESULTS, "w"))
            print(f"n={n:2d} draw={draw} [{inp:5s}] {name:28s} cv={cv_acc:.3f} test={rec['test_acc']:.3f} "
                  f"({rec['secs']}s) {rec['hp']}", flush=True)
print("done:", len(results), "records in", RESULTS)


In [ ]:
# ---- report: summary table, paired tests, figure ----------------------------------------
import json, numpy as np, pandas as pd
from scipy.stats import binomtest
import matplotlib.pyplot as plt

R = pd.DataFrame(json.load(open(RESULTS)).values())
R = R[~R.model.isin(SKIP_MODELS)]          # partial fits of skipped models stay in the JSON but are not reported
R["label"] = R["model"] + " [" + R["input"] + "]"
# Every model receives the same input x. The quantum pipelines apply Shor's discrete-log circuit to x inside the model
# (simulated here with a lookup table); "log x" in the raw records marks where that step was applied, not a different input.
DISPLAY = {
    # thesis table and figure: the uncapped setup (walk resolution h in {3..n}, LAT kernel over every width)
    "QW-QNN (continuous-time walk, full resolution) [log x]": ("main", "QW-QNN: Shor DLP -> continuous-time walk -> readout"),
    "LAT kernel (full grid) [log x]": ("main", "LAT: Shor DLP -> quantum kernel SVM"),
    "MLP [x]": ("main", "MLP"), "SVC (RBF) [x]": ("main", "SVM (RBF)"), "Random forest [x]": ("main", "Random forest"),
    "IQP kernel [x]": ("main", "IQP kernel"), "Projected quantum kernel [x]": ("main", "Projected quantum kernel"),
    "Data re-uploading [x]": ("main", "Data re-uploading"),
    "No-walk control (full resolution) [log x]": ("ablation", "Shor DLP -> no walk (t = 0) -> readout"),
    "MLP (reference) [log x]": ("ablation", "log x given (not classically efficient) -> MLP"),
    "SVC (RBF) (reference) [log x]": ("ablation", "log x given (not classically efficient) -> SVM (RBF)"),
    "Random forest (reference) [log x]": ("ablation", "log x given (not classically efficient) -> Random forest"),
    "IQP kernel [log x]": ("ablation", "Shor DLP -> IQP kernel"),
    "Projected quantum kernel [log x]": ("ablation", "Shor DLP -> Projected quantum kernel"),
    "Data re-uploading [log x]": ("ablation", "Shor DLP -> Data re-uploading"),
    # earlier capped runs (h <= 6, narrow LAT grid): kept in the JSON and README, not in the thesis outputs
    "QW-QNN (Hadamard walk) [log x]": ("omitted", "Hadamard walk (capped)"),
    "QW-QNN (continuous-time walk) [log x]": ("omitted", "continuous-time walk (capped)"),
    "LAT kernel [log x]": ("omitted", "LAT kernel (narrow grid)"),
    "No-walk control (t=0) [log x]": ("omitted", "no walk (capped)"),
}
R["table"] = R.label.map(lambda l: DISPLAY[l][0]); R["name"] = R.label.map(lambda l: DISPLAY[l][1])
ORDER = [v[1] for v in DISPLAY.values()]
summ = R.groupby(["table", "name", "n"])["test_acc"].agg(["mean", "std", "count"]).reset_index().fillna({"std": 0.0})

def fmt_table(which):
    s = summ[summ.table == which]
    m = s.pivot_table(index="name", columns="n", values="mean"); sd = s.pivot_table(index="name", columns="n", values="std")
    out = m.map(lambda v: f"{v:.3f}") + " ± " + sd.map(lambda v: f"{v:.3f}")
    return out.reindex([o for o in ORDER if o in out.index])

fmt, fmt_abl = fmt_table("main"), fmt_table("ablation")
print("Main comparison. Every model receives x; mean test accuracy ± s.d. over draws (columns: bits n of p)\n")
print(fmt.to_string())
print("\nAblations: what the discrete-log step and the walk each contribute\n")
print(fmt_abl.to_string())

def mcnemar(a, b, n):
    """Exact McNemar test on test points pooled over all draws of size n (model a vs model b)."""
    A = R[(R.label == a) & (R.n == n)].sort_values("draw"); B = R[(R.label == b) & (R.n == n)].sort_values("draw")
    if len(A) == 0 or len(A) != len(B):
        return None
    ca = np.concatenate(A.correct.values); cb = np.concatenate(B.correct.values)
    a_only, b_only = int(((ca == 1) & (cb == 0)).sum()), int(((ca == 0) & (cb == 1)).sum())
    pval = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    return dict(n=n, a=DISPLAY[a][1], b=DISPLAY[b][1], acc_a=ca.mean(), acc_b=cb.mean(), a_only=a_only, b_only=b_only, p=pval)

QW = "QW-QNN (continuous-time walk, full resolution) [log x]"   # main model; the Hadamard-walk variant is reported in the README only
tests = []
for n in sorted(R.n.unique()):
    best_cl_x = (R[(R.n == n) & R.label.isin(["MLP [x]", "SVC (RBF) [x]", "Random forest [x]"])]
                 .groupby("label").test_acc.mean().idxmax())
    for other in [best_cl_x, "LAT kernel (full grid) [log x]", "No-walk control (full resolution) [log x]"]:
        t = mcnemar(QW, other, n)
        if t and t not in tests:
            tests.append(t)
T = pd.DataFrame(tests).drop_duplicates(subset=["n", "a", "b"])
print("\nPaired exact McNemar tests, QW-QNN vs comparator (test points pooled over draws):\n")
print(T.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

shots = R[R.label == QW].copy()
if "test_acc_shots" in shots:
    for S in ["100", "1000"]:
        shots[f"shots={S}"] = shots.test_acc_shots.map(lambda d: d[S] if isinstance(d, dict) else np.nan)
    print("\nQW-QNN (continuous-time walk) with finite measurement shots (mean over draws):")
    print(shots.groupby("n")[["test_acc", "shots=100", "shots=1000"]].mean().round(3).to_string())
print("\nQW-QNN (continuous-time walk) selected hyperparameters:")
print(R[R.label == QW][["n", "draw", "hp"]].to_string(index=False))

# figure: mean test accuracy vs problem size
SER = [(QW, "QW-QNN: Shor DLP → continuous-time walk", "#2a78d6", "o"),
       ("LAT kernel (full grid) [log x]", "LAT: Shor DLP → quantum kernel", "#eb6834", "s")]
fig, ax = plt.subplots(figsize=(6.4, 4.0), dpi=150)
ns = sorted(R.n.unique())
for lab, name, col, mk in SER:
    s = summ[summ.name == DISPLAY[lab][1]].set_index("n")
    ax.errorbar(s.index, s["mean"], yerr=s["std"], color=col, marker=mk, ms=6, lw=2, capsize=3, label=name)
CLASSICAL = ["MLP", "SVC (RBF)", "Random forest"]
best_x = R[(R.input == "x") & R.model.isin(CLASSICAL)].groupby(["n", "label"]).test_acc.mean().groupby("n").max()
ax.plot(best_x.index, best_x.values, color="#eda100", marker="D", ms=6, lw=2, ls="--",
        label="Best classical model (MLP / SVM / RF)")
ax.axhline(0.5, color="#8c8c86", lw=1, ls=":")
ax.text(ns[0] + 0.1, 0.515, "chance", ha="left", va="bottom", fontsize=8, color="#5f5e58")
ax.set_xticks(ns); ax.set_xlabel("bits n of the prime p"); ax.set_ylabel("test accuracy")
ax.set_ylim(0.3, 1.02); ax.grid(axis="y", color="#e6e5df", lw=0.8); ax.set_axisbelow(True)
for sp in ["top", "right"]:
    ax.spines[sp].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="center right")
ax.set_title("Discrete-log concept class (Liu–Arunachalam–Temme); every model receives x", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "fig_lat_accuracy.png")); fig.savefig(os.path.join(OUT_DIR, "fig_lat_accuracy.pdf"))
fmt.to_csv(os.path.join(OUT_DIR, "lat_summary.csv")); fmt_abl.to_csv(os.path.join(OUT_DIR, "lat_ablation.csv")); T.to_csv(os.path.join(OUT_DIR, "lat_mcnemar.csv"), index=False)
plt.show()

# LaTeX tables for the thesis (thesis/tab_lat_main.tex, thesis/tab_lat_ablation.tex): percentages, mean ± s.d.
TEX_NAMES = {
    "QW-QNN: Shor DLP -> Hadamard walk -> readout": r"QW-QNN: DLP $\to$ Hadamard walk $\to$ read-out",
    "QW-QNN: Shor DLP -> continuous-time walk -> readout": r"Walk QNN: DLP $\to$ CTQW $\to$ read-out",
    "LAT: Shor DLP -> quantum kernel SVM": r"LAT: DLP $\to$ kernel SVM",
    "MLP": "MLP", "SVM (RBF)": "SVM (RBF)", "Random forest": "Random forest",
    "Shor DLP -> no walk (t = 0) -> readout": r"DLP $\to$ no walk $\to$ read-out",
    "log x given (not classically efficient) -> MLP": r"$\log_g x$ given $\to$ MLP",
    "log x given (not classically efficient) -> SVM (RBF)": r"$\log_g x$ given $\to$ SVM (RBF)",
    "log x given (not classically efficient) -> Random forest": r"$\log_g x$ given $\to$ random forest",
}

def tex_table(which, fname, label, caption, short):
    s = summ[summ.table == which]
    ns_ = sorted(s.n.unique())
    rows = [o for o in ORDER if o in set(s.name)]
    lines = [r"\begin{table}[htbp]", r"\centering", r"\small", r"\setlength{\tabcolsep}{4pt}", rf"\caption[{short}]{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{l" + "r" * len(ns_) + "}", r"\hline",
             "Model & " + " & ".join(f"$n={n}$" for n in ns_) + r" \\", r"\hline"]
    for r in rows:
        cells = []
        for n in ns_:
            v = s[(s.name == r) & (s.n == n)].iloc[0]
            cells.append(f"${100 * v['mean']:.1f} \\pm {100 * v['std']:.1f}$")
        lines.append(TEX_NAMES.get(r, r) + " & " + " & ".join(cells) + r" \\")
    lines += [r"\hline", r"\end{tabular}", r"\end{table}", ""]
    open(os.path.join(OUT_DIR, fname), "w").write("\n".join(lines))

tex_table("main", "tab_lat_main.tex", "tab:latmain",
          r"Discrete-logarithm task: test accuracy (\%) over five draws; every model receives $x$, and $n$ is the bit length of $p$. DLP: Shor's discrete-logarithm step; CTQW: continuous-time quantum walk.",
          "Discrete-logarithm task: main comparison")
tex_table("ablation", "tab_lat_ablation.tex", "tab:latablation",
          r"Discrete-logarithm task, ablations: the walk removed, and classical models supplied with $\log_g x$, which no efficient classical algorithm computes for large $n$. Test accuracy (\%) over five draws.",
          "Discrete-logarithm task: ablations")
